
# **ML Modelling**

In [3]:
# LOAD DATA
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob, os, joblib

folders = glob.glob("/content/drive/MyDrive/Time_data_after_corr_*")

if len(folders) == 0:
    raise ValueError("No saved dataset found. Check your folder name.")

latest_folder = max(folders, key=os.path.getmtime)
print("Loading data from:", latest_folder)

# FIX: correct filenames
X_train = joblib.load(os.path.join(latest_folder, "X_train_final.pkl"))
X_test = joblib.load(os.path.join(latest_folder, "X_test_final.pkl"))
y_train = joblib.load(os.path.join(latest_folder, "y_train.pkl"))
y_test = joblib.load(os.path.join(latest_folder, "y_test.pkl"))


# LOAD FEATURE SELECTION RESULTS
fs_folders = glob.glob("/content/drive/MyDrive/Time_FS_results_*")

if len(fs_folders) == 0:
    raise ValueError("No FS results found.")

latest_fs_folder = max(fs_folders, key=os.path.getmtime)
print("Loading FS from:", latest_fs_folder)

fs_methods = {}

for file in os.listdir(latest_fs_folder):
    if file.endswith(".pkl"):
        name = file.replace(".pkl", "")
        fs_methods[name] = joblib.load(os.path.join(latest_fs_folder, file))

# ADD full dataset (no FS)
fs_methods["ALL_FEATURES"] = X_train.columns.tolist()


# SMOTE (TRAIN ONLY)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Import
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# EVALUATION METRICS FUNCTION
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred) * 100,
        "Precision (W)": precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Recall (W)": recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "F1 (W)": f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Precision (Macro)": precision_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "Recall (Macro)": recall_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "F1 (Macro)": f1_score(y_test, y_pred, average='macro', zero_division=0) * 100,
    }

def print_results(results):
    for k, v in results.items():
        print(f"{k}: {v:.2f}%")

Mounted at /content/drive
Loading data from: /content/drive/MyDrive/Time_data_after_corr_20260507_1552
Loading FS from: /content/drive/MyDrive/Time_FS_results_20260507_1552


In [ ]:
# Print Loaded Feature Sets

print("\n================ FEATURE SETS LOADED ================")

for method, feats in fs_methods.items():
    print(f"\n{method}:")
    print(f"Number of features: {len(feats)}")
    print("Features:", feats)



================ FEATURE SETS LOADED ================

ANOVA:
Number of features: 15
Features: ['Weekly_Exercise_Duration', 'Risk Factor - ECHO - EF', 'Risk Factor - Stress', 'Risk Factor - HPL', 'Risk Factor - Exercise', 'Occupation', 'Total_Muscle_Power', 'Predicted Risk Level Encoded', 'Year', 'Posture', 'Exercise Habit - Mode', 'Balance in Sitting and Standing', 'Walking', 'RISK  - Risk Type', 'ECG Resting']

Mutual_Info:
Number of features: 15
Features: ['Weekly_Exercise_Duration', 'Risk Factor - Stress', 'Risk Factor - ECHO - EF', 'Risk Factor - BMI', 'Test Today - METS', 'Risk Factor - Exercise', 'Risk Factor - Family hx', 'Cooling Down', 'Total_Muscle_Power', 'ROM', 'Functional Activity', 'Living Environment', 'Year', 'Gender', 'RISK  - Risk Type']

RFE_LR:
Number of features: 15
Features: ['Weekly_Exercise_Duration', 'Exercise Habit - Frequency', 'Posture', 'Lives With', 'Risk Factor - HPL', 'Occupation', 'Risk Factor - ECHO - EF', 'Risk Factor - Stress', 'Smoking', 'Risk Fac

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# **Conventional Models**

*   Decision Tree
*   Logistic Regression (Classifier)
*   k-Nearest Neighbors (k-NN)



In [ ]:
# CONVENTIONAL MODELS

import numpy as np
import random
import os
import warnings

warnings.filterwarnings("ignore")

from collections import Counter

from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================================================
# FIXED SEED
# =========================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================================================
# SAFE CV
# =========================================================

min_class_count = min(Counter(y_train).values())

n_splits = min(5, min_class_count)
n_splits = max(2, n_splits)

print(f"Using {n_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED
)

# =========================================================
# MODELS
# =========================================================

models = {

    "Decision Tree": (

        DecisionTreeClassifier(
            class_weight='balanced',
            random_state=SEED
        ),

        {
            "max_depth": list(range(2, 30))
        }
    ),

    "Logistic Regression": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("model", LogisticRegression(
                max_iter=5000,
                random_state=SEED
            ))
        ]),

        {
            "model__C": np.logspace(-3, 2, 10)
        }
    ),

    "k-NN": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("model", KNeighborsClassifier())
        ]),

        {
            "model__n_neighbors": list(range(3, 15))
        }
    )
}

# =========================================================
# LOOP
# =========================================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== CONVENTIONAL | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train.loc[:, features]
    X_test_sel = X_test.loc[:, features]

    for name, (model, params) in models.items():

        print(f"\n{name}")

        total_space = int(
            np.prod([len(v) for v in params.values()])
        )

        n_iter = min(10, total_space)

        search = RandomizedSearchCV(

            estimator=model,

            param_distributions=params,

            n_iter=n_iter,

            cv=cv,

            scoring='accuracy',

            n_jobs=1,

            random_state=SEED
        )

        # TRAIN
        search.fit(X_train_sel, y_train)

        # TEST
        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)

        print_results(
            evaluate(y_test, y_pred)
        )

Using 5-Fold Stratified CV


========== CONVENTIONAL | FEATURE SET: ANOVA ==========

Decision Tree
Best Params: {'max_depth': 2}
Accuracy: 72.22%
Precision (W): 74.88%
Recall (W): 72.22%
F1 (W): 72.38%
Precision (Macro): 76.39%
Recall (Macro): 69.30%
F1 (Macro): 71.62%

Logistic Regression
Best Params: {'model__C': np.float64(2.1544346900318843)}
Accuracy: 50.00%
Precision (W): 49.90%
Recall (W): 50.00%
F1 (W): 47.91%
Precision (Macro): 49.60%
Recall (Macro): 54.99%
F1 (Macro): 50.44%

k-NN
Best Params: {'model__n_neighbors': 7}
Accuracy: 38.89%
Precision (W): 42.86%
Recall (W): 38.89%
F1 (W): 40.19%
Precision (Macro): 40.48%
Recall (Macro): 38.17%
F1 (Macro): 38.56%


========== CONVENTIONAL | FEATURE SET: Mutual_Info ==========

Decision Tree
Best Params: {'max_depth': 2}
Accuracy: 72.22%
Precision (W): 74.88%
Recall (W): 72.22%
F1 (W): 72.38%
Precision (Macro): 76.39%
Recall (Macro): 69.30%
F1 (Macro): 71.62%

Logistic Regression
Best Params: {'model__C': np.float64(7.7426368268112

# **Ensemble models**

*   CatBoost
*   LightGBM
*   XGBoost


In [ ]:
# ENSEMBLE MODELS

import numpy as np
import random
import os

from collections import Counter

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# INSTALL MISSING PACKAGES
import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# CatBoost
try:
    from catboost import CatBoostClassifier
except ModuleNotFoundError:
    install("catboost")
    from catboost import CatBoostClassifier

# LightGBM
try:
    from lightgbm import LGBMClassifier
except ModuleNotFoundError:
    install("lightgbm")
    from lightgbm import LGBMClassifier

# XGBoost
try:
    import xgboost as xgb
except ModuleNotFoundError:
    install("xgboost")
    import xgboost as xgb

# ============================================
# FIXED SEED
# ============================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================
# SAFE CV
# ============================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# ============================================
# MODELS
# ============================================

ensembles = {

    "Random Forest": (
        RandomForestClassifier(
            class_weight='balanced',
            random_state=SEED,
            n_jobs=1
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
        }
    ),

    "LightGBM": (
        LGBMClassifier(
            class_weight='balanced',
            random_state=SEED,
            verbosity=-1,
            n_jobs=1
        ),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
        }
    ),

    "XGBoost": (
        xgb.XGBClassifier(
            eval_metric='mlogloss',
            random_state=SEED,
            use_label_encoder=False,
            n_jobs=1
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [3, 5],
            "learning_rate": [0.05, 0.1],
        }
    ),

    "CatBoost": (
        CatBoostClassifier(
            auto_class_weights="Balanced",
            verbose=0,
            random_seed=SEED
        ),
        {
            "iterations": [200, 400],
            "depth": [4, 6],
            "learning_rate": [0.05, 0.1],
        }
    ),
}

# ============================================
# MAIN LOOP
# ============================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== ENSEMBLE | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    for name, (model, params) in ensembles.items():

        print(f"\n{name}")

        total_space = int(np.prod([len(v) for v in params.values()]))
        n_iter = min(10, total_space)

        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=n_iter,
            cv=cv,
            scoring='accuracy',
            n_jobs=1,
            random_state=SEED
        )

        search.fit(X_train_sel, y_train)

        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)
        print_results(evaluate(y_test, y_pred))

Using 5-Fold Stratified CV


========== ENSEMBLE | FEATURE SET: ANOVA ==========

Random Forest
Best Params: {'n_estimators': 200, 'max_depth': 10}
Accuracy: 52.78%
Precision (W): 54.17%
Recall (W): 52.78%
F1 (W): 53.01%
Precision (Macro): 54.17%
Recall (Macro): 50.35%
F1 (Macro): 51.79%

LightGBM
Best Params: {'n_estimators': 200, 'learning_rate': 0.05}
Accuracy: 55.56%
Precision (W): 54.70%
Recall (W): 55.56%
F1 (W): 54.83%
Precision (Macro): 53.34%
Recall (Macro): 54.05%
F1 (Macro): 53.32%

XGBoost
Best Params: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
Accuracy: 58.33%
Precision (W): 55.52%
Recall (W): 58.33%
F1 (W): 54.58%
Precision (Macro): 55.09%
Recall (Macro): 52.53%
F1 (Macro): 51.26%

CatBoost
Best Params: {'learning_rate': 0.05, 'iterations': 200, 'depth': 4}
Accuracy: 61.11%
Precision (W): 63.16%
Recall (W): 61.11%
F1 (W): 61.81%
Precision (Macro): 62.87%
Recall (Macro): 59.72%
F1 (Macro): 60.91%


========== ENSEMBLE | FEATURE SET: Mutual_Info ==========


# **Deep learning**

*   FCN
*   BPNN
*   CNN







In [ ]:
# DEEP LEARNING MODELS

import numpy as np
import tensorflow as tf
import random
import os

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# FIXED SEED
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

os.environ['TF_DETERMINISTIC_OPS'] = '1'

# SETTINGS
EPOCHS = 40
BATCH_SIZE = 32

# CLASS WEIGHTS
def get_class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    return dict(zip(classes, weights))

# MODEL DEFINITIONS
def build_fcn(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def build_bpnn(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def build_cnn(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Reshape((input_dim, 1)),
        tf.keras.layers.Conv1D(16, 3, padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

# TRAINING FUNCTIONS
def train_standard(model_fn, X_train, y_train, X_test, y_test):

    num_classes = len(np.unique(y_train))
    input_dim = X_train.shape[1]

    model = model_fn(input_dim, num_classes)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    class_weights = get_class_weights(y_train)

    model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        class_weight=class_weights,
        shuffle=True
    )

    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    return evaluate(y_test, y_pred)


# MAIN LOOP
for fs_name, features in fs_methods.items():

    print(f"\n\n========== DEEP LEARNING | FEATURE SET: {fs_name} ==========")

    # SCALE
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train[features])
    X_test_s = scaler.transform(X_test[features])

    # FCN
    print("\nFCN")
    print_results(train_standard(build_fcn, X_train_s, y_train, X_test_s, y_test))

    # BPNN
    print("\nBPNN")
    print_results(train_standard(build_bpnn, X_train_s, y_train, X_test_s, y_test))

    # CNN
    print("\nCNN")
    print_results(train_standard(build_cnn, X_train_s, y_train, X_test_s, y_test))



========== DEEP LEARNING | FEATURE SET: ANOVA ==========

FCN
Accuracy: 63.89%
Precision (W): 64.13%
Recall (W): 63.89%
F1 (W): 63.94%
Precision (Macro): 62.92%
Recall (Macro): 63.79%
F1 (Macro): 63.27%

BPNN
Accuracy: 58.33%
Precision (W): 58.02%
Recall (W): 58.33%
F1 (W): 58.11%
Precision (Macro): 57.04%
Recall (Macro): 56.38%
F1 (Macro): 56.64%

CNN


Accuracy: 55.56%
Precision (W): 54.63%
Recall (W): 55.56%
F1 (W): 54.88%
Precision (Macro): 54.02%
Recall (Macro): 54.05%
F1 (Macro): 53.77%

Autoencoder Classifier
Accuracy: 52.78%
Precision (W): 52.34%
Recall (W): 52.78%
F1 (W): 51.51%
Precision (Macro): 51.52%
Recall (Macro): 47.97%
F1 (Macro): 48.70%


========== DEEP LEARNING | FEATURE SET: Mutual_Info ==========

FCN
Accuracy: 41.67%
Precision (W): 40.51%
Recall (W): 41.67%
F1 (W): 40.27%
Precision (Macro): 38.96%
Recall (Macro): 37.02%
F1 (Macro): 37.19%

BPNN
Accuracy: 47.22%
Precision (W): 47.00%
Recall (W): 47.22%
F1 (W): 47.05%
Precision (Macro): 45.30%
Recall (Macro): 45.42%
F1 (Macro): 45.28%

CNN
Accuracy: 47.22%
Precision (W): 46.95%
Recall (W): 47.22%
F1 (W): 46.89%
Precision (Macro): 46.10%
Recall (Macro): 45.42%
F1 (Macro): 45.52%

Autoencoder Classifier
Accuracy: 58.33%
Precision (W): 58.46%
Recall (W): 58.33%
F1 (W): 57.16%
Precision (Macro): 58.59%
Recall (Macro): 53.64%
F1 (Macro): 54.81%


========== DEEP LEARNIN

# **Transformer models**

*   FT-Transformer
*   TabTransformer
*   TabNet


In [ ]:
# TRANSFORMER MODELS

import numpy as np
import random
import os
import warnings

warnings.filterwarnings("ignore")

from collections import Counter

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier

from sklearn.ensemble import HistGradientBoostingClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================================================
# FIXED SEED
# =========================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================================================
# SAFE CV
# =========================================================

min_class_count = min(Counter(y_train).values())

n_splits = min(5, min_class_count)
n_splits = max(2, n_splits)

print(f"Using {n_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED
)

# =========================================================
# MODELS
# =========================================================

transformer_models = {

    # FT-Transformer Approximation
    "FT-Transformer (Approx)": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("mlp", MLPClassifier(
                hidden_layer_sizes=(256, 128, 64),
                max_iter=1500,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=SEED
            ))
        ]),

        {
            "mlp__alpha": [0.0001, 0.001]
        }
    ),

    # TabTransformer Approximation
    "TabTransformer (Approx)": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("mlp", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                max_iter=1200,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=SEED
            ))
        ]),

        {
            "mlp__alpha": [0.0001, 0.001]
        }
    ),
}

# =========================================================
# LOOP
# =========================================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== TRANSFORMER | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train.loc[:, features]
    X_test_sel = X_test.loc[:, features]

    for name, (model, params) in transformer_models.items():

        print(f"\n{name}")

        total_space = int(
            np.prod([len(v) for v in params.values()])
        )

        n_iter = min(10, total_space)

        search = RandomizedSearchCV(

            estimator=model,

            param_distributions=params,

            n_iter=n_iter,

            cv=cv,

            scoring='accuracy',

            n_jobs=1,

            random_state=SEED
        )

        # TRAIN
        search.fit(X_train_sel, y_train)

        # TEST
        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)

        print_results(
            evaluate(y_test, y_pred)
        )

Using 5-Fold Stratified CV


========== TRANSFORMER | FEATURE SET: ANOVA ==========

FT-Transformer (Approx)
Best Params: {'mlp__alpha': 0.0001}
Accuracy: 38.89%
Precision (W): 39.31%
Recall (W): 38.89%
F1 (W): 33.91%
Precision (Macro): 39.17%
Recall (Macro): 46.14%
F1 (Macro): 38.11%

TabTransformer (Approx)
Best Params: {'mlp__alpha': 0.0001}
Accuracy: 47.22%
Precision (W): 48.87%
Recall (W): 47.22%
F1 (W): 46.50%
Precision (Macro): 47.46%
Recall (Macro): 50.65%
F1 (Macro): 47.62%

Attention-like Boosting (HistGB)
Best Params: {'max_depth': 5, 'learning_rate': 0.05}
Accuracy: 58.33%
Precision (W): 56.92%
Recall (W): 58.33%
F1 (W): 56.96%
Precision (Macro): 56.08%
Recall (Macro): 54.27%
F1 (Macro): 54.50%


========== TRANSFORMER | FEATURE SET: Mutual_Info ==========

FT-Transformer (Approx)
Best Params: {'mlp__alpha': 0.0001}
Accuracy: 38.89%
Precision (W): 40.72%
Recall (W): 38.89%
F1 (W): 38.50%
Precision (Macro): 39.60%
Recall (Macro): 41.29%
F1 (Macro): 39.27%

TabTransformer (Ap

In [5]:
# TABNET MODEL

import numpy as np
import random
import os

from collections import Counter

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

# INSTALL TABNET IF MISSING
import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    from pytorch_tabnet.tab_model import TabNetClassifier
except ModuleNotFoundError:
    install("pytorch-tabnet")
    from pytorch_tabnet.tab_model import TabNetClassifier

# ============================================
# FIXED SEED
# ============================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================
# SAFE CV
# ============================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# ============================================
# TABNET WRAPPER
# ============================================

class TabNetWrapper(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        n_d=16,
        n_a=16,
        n_steps=3,
        gamma=1.3,
        lambda_sparse=1e-4
    ):

        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse

    def fit(self, X, y):

        self.model_ = TabNetClassifier(
            n_d=self.n_d,
            n_a=self.n_a,
            n_steps=self.n_steps,
            gamma=self.gamma,
            lambda_sparse=self.lambda_sparse,
            seed=SEED,
            verbose=0
        )

        self.model_.fit(
            X_train=X.values if hasattr(X, "values") else X,
            y_train=y,
            max_epochs=100,
            batch_size=32,
            virtual_batch_size=16
        )

        return self

    def predict(self, X):

        return self.model_.predict(
            X.values if hasattr(X, "values") else X
        )

    def predict_proba(self, X):

        return self.model_.predict_proba(
            X.values if hasattr(X, "values") else X
        )

# ============================================
# PIPELINE
# ============================================

tabnet_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", TabNetWrapper())
])

# ============================================
# PARAMETER SPACE
# ============================================

param_dist = {

    "model__n_d": [8, 16, 32],

    "model__n_a": [8, 16, 32],

    "model__n_steps": [3, 5],

    "model__gamma": [1.0, 1.3],

    "model__lambda_sparse": [1e-3, 1e-4]
}

# ============================================
# MAIN LOOP
# ============================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== TABNET | FEATURE SET: {fs_name} ==========")

    # FEATURE SELECTION
    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    # RANDOM SEARCH
    total_space = int(np.prod([len(v) for v in param_dist.values()]))

    n_iter = min(10, total_space)

    search = RandomizedSearchCV(
        estimator=tabnet_pipeline,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring='accuracy',
        n_jobs=1,
        random_state=SEED
    )

    # TRAIN
    search.fit(X_train_sel, y_train)

    # TEST
    y_pred = search.predict(X_test_sel)

    # RESULTS
    print("Best Params:", search.best_params_)

    print_results(
        evaluate(y_test, y_pred)
    )

Using 5-Fold Stratified CV


========== TABNET | FEATURE SET: ANOVA ==========


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/

Best Params: {'model__n_steps': 3, 'model__n_d': 32, 'model__n_a': 8, 'model__lambda_sparse': 0.001, 'model__gamma': 1.0}
Accuracy: 47.22%
Precision (W): 41.19%
Recall (W): 47.22%
F1 (W): 43.89%
Precision (Macro): 36.47%
Recall (Macro): 42.94%
F1 (Macro): 39.34%


========== TABNET | FEATURE SET: Mutual_Info ==========


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/

Best Params: {'model__n_steps': 3, 'model__n_d': 32, 'model__n_a': 16, 'model__lambda_sparse': 0.001, 'model__gamma': 1.0}
Accuracy: 44.44%
Precision (W): 46.33%
Recall (W): 44.44%
F1 (W): 44.59%
Precision (Macro): 46.43%
Recall (Macro): 41.72%
F1 (Macro): 43.24%


========== TABNET | FEATURE SET: RFE_LR ==========


...

Best Params: {'model__n_steps': 3, 'model__n_d': 8, 'model__n_a': 32, 'model__lambda_sparse': 0.001, 'model__gamma': 1.0}
Accuracy: 72.22%
Precision (W): 69.26%
Recall (W): 72.22%
F1 (W): 69.72%
Precision (Macro): 68.89%
Recall (Macro): 68.19%
F1 (Macro): 67.42%


========== TABNET | FEATURE SET: RF ==========


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/

Best Params: {'model__n_steps': 3, 'model__n_d': 32, 'model__n_a': 32, 'model__lambda_sparse': 0.0001, 'model__gamma': 1.0}
Accuracy: 72.22%
Precision (W): 72.46%
Recall (W): 72.22%
F1 (W): 68.75%
Precision (Macro): 73.96%
Recall (Macro): 66.82%
F1 (Macro): 66.11%


========== TABNET | FEATURE SET: ALL_FEATURES ==========


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/

Best Params: {'model__n_steps': 3, 'model__n_d': 8, 'model__n_a': 32, 'model__lambda_sparse': 0.001, 'model__gamma': 1.0}
Accuracy: 72.22%
Precision (W): 75.09%
Recall (W): 72.22%
F1 (W): 72.77%
Precision (Macro): 76.14%
Recall (Macro): 71.05%
F1 (Macro): 72.58%
